# Regression. Part 2

---


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [2]:
def metrics(true, pred):
    print('R2:', r2_score(true, pred))
    print('MAE:', mean_absolute_error(true, pred))
    print('RMSE:', mean_squared_error(true, pred)**0.5)

In [4]:
df = pd.read_csv('house_price_regression_dataset.csv')

In [5]:
df.head()

,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
0,1360,2,1,1981,0.599637,0,5,2.623829e+05
1,4272,3,3,2016,4.753014,1,6,9.852609e+05
2,3592,1,2,2016,3.634823,0,9,7.779774e+05
3,966,1,2,1977,2.730667,1,8,2.296989e+05
4,4926,2,1,1993,4.699073,0,8,1.041741e+06


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Square_Footage        1000 non-null   int64  
 1   Num_Bedrooms          1000 non-null   int64  
 2   Num_Bathrooms         1000 non-null   int64  
 3   Year_Built            1000 non-null   int64  
 4   Lot_Size              1000 non-null   float64
 5   Garage_Size           1000 non-null   int64  
 6   Neighborhood_Quality  1000 non-null   int64  
 7   House_Price           1000 non-null   float64
dtypes: float64(2), int64(6)
memory usage: 62.6 KB


In [7]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(['House_Price'], axis=1), df['House_Price'], test_size=0.2, random_state=42)

# Модели

In [8]:
from sklearn.linear_model import LinearRegression

In [9]:
lr = LinearRegression().fit(X_train, y_train)

In [10]:
metrics(y_test, lr.predict(X_test))

R2: 0.9984263636823408
MAE: 8174.583600008702
RMSE: 10071.484424138667


In [11]:
from sklearn.ensemble import RandomForestRegressor

In [12]:
rf = RandomForestRegressor().fit(X_train, y_train)

In [13]:
metrics(y_test, rf.predict(X_test))

R2: 0.9938367772119996
MAE: 16049.030428751013
RMSE: 19931.740939572002


In [14]:
from sklearn.ensemble import GradientBoostingRegressor

In [15]:
gbr = GradientBoostingRegressor().fit(X_train, y_train)

In [16]:
metrics(y_test, gbr.predict(X_test))

R2: 0.9965087150085455
MAE: 12307.185036973115
RMSE: 15001.474604564892


In [ ]:
from catboost import CatBoostRegressor

In [ ]:
cb = CatBoostRegressor().fit(X_train, y_train, verbose=False)

In [ ]:
metrics(y_test, cb.predict(X_test))

# Отбор признаков

## Прямой отбор

In [19]:
from sklearn.feature_selection import SequentialFeatureSelector

In [ ]:
cb = CatBoostRegressor(verbose=False)

In [ ]:
sfs.get_support()

In [ ]:
sfs.get_params()

In [ ]:
sfs.transform(X_test)

## Последовательный отбор

In [ ]:
cb = CatBoostRegressor(verbose=False)
sfs = SequentialFeatureSelector(cb, direction='backward')
sfs.fit(X_train, y_train)

In [ ]:
sfs.get_support()

## Исчерпывающий выбор

In [ ]:
from mlxtend.feature_selection import ExhaustiveFeatureSelector

In [ ]:
cb = CatBoostRegressor(verbose=False)
efs = ExhaustiveFeatureSelector(cb, min_features=1, max_features=7, scoring='r2', cv=5)
efs.fit(X_train, y_train)

In [ ]:
efs.best_score_

In [ ]:
efs.best_feature_names_

In [ ]:
efs.subsets_

# Pipeline

## Pipeline as transformer

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler

In [ ]:
simple_imputer = SimpleImputer(strategy='mean')
scaler = MinMaxScaler()

In [ ]:
pipe = Pipeline(steps=[('imputer', simple_imputer), ('scaler', scaler)])

In [ ]:
pipe.fit(X_train)

In [ ]:
pipe.transform(X_test)

## Pipeline as model

In [ ]:
model = LinearRegression()

In [ ]:
pipe = Pipeline(steps=[('imputer', simple_imputer), ('scaler', scaler), ('model', model)])

In [ ]:
pipe.fit(X_train, y_train)

In [ ]:
metrics(y_test, pipe.predict(X_test))

## Обработка разнородных данных

In [ ]:
df = pd.read_csv('insurance.csv')

In [ ]:
df.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(['charges'], axis=1), df['charges'], test_size=0.2, random_state=42)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [ ]:
col_transformer = ColumnTransformer([('num_preproc', MinMaxScaler(), [x for x in X_train.columns if X_train[x].dtype!='object']),
                                     ('cat_preproc', OneHotEncoder(dtype='int'), [x for x in X_train.columns if X_train[x].dtype=='object'])])

In [ ]:
pipe = Pipeline([('preproc', col_transformer), ('LR', LinearRegression())])

In [ ]:
pipe.fit(X_train, y_train)

In [ ]:
metrics(y_test, pipe.predict(X_test))

## Подбор гиперпараметров

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
pipe = Pipeline([('preproc', col_transformer), ('CatBoost', CatBoostRegressor(verbose=False))])

In [ ]:
pipe.fit(X_train, y_train)

In [ ]:
pipe.score(X_test, y_test)

In [ ]:
param_grid = {
    "CatBoost__iterations": [1000, 2000],
    "CatBoost__learning_rate": [0.01, 0.05],
    "CatBoost__depth": [3, 5, 7]
}
search = GridSearchCV(pipe, param_grid)

In [ ]:
search.fit(X_train, y_train)

In [ ]:
search.best_score_

In [ ]:
search.best_params_